# FellaRide: the ML models, how they're trained, and how the app uses them

This notebook covers **every learning part of FellaRide**, end to end, and runs top to bottom in about a minute.

| Part | What it learns | Method | Where |
|---|---|---|---|
| **Propensity scorer** | How likely a member is to act as a driver, rider, connector or referrer when nudged | Logistic regression per role, trained offline | `ml/train_scorer.py` → `models/scorer.pkl` |
| **Online learning** | Each community's own weights, after every nudge answer, ride and rating | One stochastic-gradient step per outcome, with a decaying learning rate | `app/engine/scorer.py` (`learn`) |
| **Expansion engine** | Which growth action works in this community | Beta(1, 1) posterior per action type plus an exploration bonus (a bandit) | `app/engine/expansion.py` |

Matching (`app/engine/matching.py`) is a weighted, interpretable **rule** (route overlap, detour, time gap, preference), not a trained model.

**Data:** all training data here is **synthetic** (labelled per hackathon rule 3). Synthetic communities are replayed through the real engine, and every nudge outcome becomes one training row. Section 8 shows the same pipeline on **real** answered nudges.

Running this notebook **does not change the shipped model**: it trains into a temporary file and checks that it reproduces `models/scorer.pkl`.

In [1]:
import sys, json, pickle, tempfile
from pathlib import Path
import numpy as np
BACKEND = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BACKEND))
from ml.train_scorer import generate_logs, load_real_logs, train, save, MIN_SAMPLES, MIN_AUC, ROLES
from app.engine.context import FEATURES
print("features:", FEATURES)
print("roles:   ", ROLES)

features: ['has_car', 'corridor_density', 'recent_rides', 'social_degree', 'past_response', 'bias']
roles:    ('driver', 'rider', 'connector', 'referrer')


## 1. Features and labels

Every model uses the same 6 features, computed by `Context.features(member)` at the moment a nudge is sent. All are scaled to 0–1.

| Feature | Formula | Meaning |
|---|---|---|
| `has_car` | 1 if the member has a car, else 0 | Can they drive at all? |
| `corridor_density` | min(1, registered members in their corridor / 6) | How many people share their home cell, destination and 30-minute window |
| `recent_rides` | min(1, rides / 4) if they rode in the last 7 days, else 0 | Current activity |
| `social_degree` | min(1, social ties / 12) | How connected they are |
| `past_response` | nudges answered / nudges received (0.5 before the first nudge) | How they reacted before |
| `bias` | always 1 | The intercept column |

**Labels**

- **driver, rider, connector:** 1 if the member acted on the nudge (offered, requested or invited), 0 if they ignored it.
- **referrer:** 1 if a registered member who hasn't referred anyone yet brings someone in during the **second half** of the run. Features are taken at the **midpoint**, before the outcome, so the label can't leak into them.

Half the runs use the smart engine and half use **random** nudges, so the data also covers members the smart engine would never pick (otherwise the model only learns about people it already likes).

In [2]:
logs = generate_logs(runs=6, days=30)
for role, rows in logs.items():
    pos = sum(y for _, y in rows)
    print(f"{role:10s} samples={len(rows):5d}  positives={pos:4d}  rate={pos/len(rows):.2f}")

driver     samples=  590  positives= 170  rate=0.29
rider      samples= 1879  positives= 543  rate=0.29
connector  samples=   51  positives=  11  rate=0.22
referrer   samples=  310  positives=  42  rate=0.14


## 2. Train one logistic regression per role

`train()` holds out 25% of rows (a random split for synthetic data), fits scikit-learn's `LogisticRegression`, and **ships a role only if it has at least 100 samples and a holdout AUC of at least 0.6**. A role that fails the bar keeps the engine's default weights.

In [3]:
models, report = train(logs)
print(f"quality bar: samples >= {MIN_SAMPLES}, holdout AUC >= {MIN_AUC}\n")
for role, r in report.items():
    print(f"{role:10s} shipped={r.get('shipped')!s:5s}  samples={r.get('samples'):5d}  positive_rate={r.get('positive_rate')}  holdout_auc={r.get('holdout_auc')}")
    if 'weights' in r: print('           weights:', r['weights'], ' intercept:', r['intercept'])

quality bar: samples >= 100, holdout AUC >= 0.6

driver     shipped=True   samples=  590  positive_rate=0.288  holdout_auc=0.776
           weights: {'has_car': -0.004, 'corridor_density': 0.344, 'recent_rides': 0.138, 'social_degree': 0.257, 'past_response': 2.213, 'bias': -0.004}  intercept: -2.088
rider      shipped=True   samples= 1879  positive_rate=0.289  holdout_auc=0.712
           weights: {'has_car': 0.0, 'corridor_density': 0.38, 'recent_rides': 0.603, 'social_degree': 0.215, 'past_response': 1.388, 'bias': 0.003}  intercept: -1.895
connector  shipped=False  samples=   51  positive_rate=0.216  holdout_auc=0.6
           weights: {'has_car': 0.357, 'corridor_density': 0.299, 'recent_rides': 0.0, 'social_degree': 0.012, 'past_response': 0.403, 'bias': -0.0}  intercept: -1.579
referrer   shipped=True   samples=  310  positive_rate=0.135  holdout_auc=0.626
           weights: {'has_car': -0.401, 'corridor_density': 1.202, 'recent_rides': 0.405, 'social_degree': -0.003, 'past_res

**Why connector isn't shipped:** it has only 51 samples, under the 100-sample minimum, and just 13 rows in the holdout, so its AUC (0.60) says almost nothing either way. The quality bar keeps the default weights for that role. That's the bar doing its job: a model trained on 51 rows shouldn't steer who gets nudged.

**Why logistic regression:** a few hundred to a few thousand rows per role, 6 features, and the weights have to be readable on the organiser's dashboard and cheap to update online after every outcome. A deep model would overfit here and couldn't be explained.

## 3. Evaluation beyond AUC

Same holdout as `train()` (same seed, same order). For each role we compare the model with:
- a **one-feature baseline**: just `past_response`, the most obvious signal
- the **base rate**, for the Brier score (lower is better; predicting the base rate for everyone is the bar to beat)

and check **calibration**: when the model says 30%, about 30% should respond.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

rng = np.random.default_rng(0)   # train() uses seed 0 and draws one permutation per role, in this order
for role, rows in logs.items():
    X = np.array([x for x, _ in rows], dtype=float); y = np.array([t for _, t in rows])
    idx = rng.permutation(len(y)); cut = int(len(y) * 0.75); tr, te = idx[:cut], idx[cut:]
    clf = LogisticRegression(max_iter=500, C=1.0).fit(X[tr], y[tr])
    p = clf.predict_proba(X[te])[:, 1]
    base_auc = roc_auc_score(y[te], X[te][:, FEATURES.index('past_response')]) if len(set(y[te])) > 1 else float('nan')
    brier, brier_base = brier_score_loss(y[te], p), brier_score_loss(y[te], np.full(len(te), y[tr].mean()))
    print(f"{role:10s} AUC model={roc_auc_score(y[te], p):.3f}  past_response alone={base_auc:.3f}   "
          f"Brier model={brier:.3f}  base rate={brier_base:.3f}  (holdout n={len(te)})")

driver     AUC model=0.776  past_response alone=0.771   Brier model=0.158  base rate=0.206  (holdout n=148)
rider      AUC model=0.712  past_response alone=0.673   Brier model=0.191  base rate=0.215  (holdout n=470)
connector  AUC model=0.600  past_response alone=0.900   Brier model=0.177  base rate=0.178  (holdout n=13)
referrer   AUC model=0.626  past_response alone=0.660   Brier model=0.128  base rate=0.131  (holdout n=78)


In [5]:
# calibration on the largest role: predicted vs actual response rate, in 4 bins of predictions
role = 'rider'
X = np.array([x for x, _ in logs[role]], dtype=float); y = np.array([t for _, t in logs[role]])
clf = models[role]
p = clf.predict_proba(X)[:, 1]
bins = np.quantile(p, [0, .25, .5, .75, 1])
print(f"{role}: predicted vs actual")
for lo, hi in zip(bins[:-1], bins[1:]):
    sel = (p >= lo) & (p <= hi)
    print(f"  predicted {p[sel].mean():.2f}   actual {y[sel].mean():.2f}   (n={sel.sum()})")

rider: predicted vs actual
  predicted 0.15   actual 0.15   (n=473)
  predicted 0.22   actual 0.19   (n=476)
  predicted 0.30   actual 0.33   (n=480)
  predicted 0.47   actual 0.48   (n=471)


**Reading the evaluation honestly**

- The model beats the **base rate** on the Brier score for driver (0.158 vs 0.206), rider (0.191 vs 0.215) and referrer (0.128 vs 0.131): its probabilities are more accurate than guessing the average. For connector they're level, which is why it isn't shipped.
- On ranking (AUC), it beats **past_response alone** clearly for **riders** (0.71 vs 0.67, the largest role) and matches it for drivers (0.78 vs 0.77).
- For **referrer**, past_response alone ranks slightly better (0.66 vs the model's 0.63) on 78 holdout rows. That gap is within noise, but it's an honest weak spot: the referrer model is the weakest of the shipped three and needs real data.
- The connector numbers rest on 13 holdout rows and shouldn't be read at all.
- **Calibration** for riders is close: in each quarter of predictions, predicted and actual response rates are within about 3 percentage points.

## 4. What the weights say

Positive weight → more likely to respond. The features are all 0–1, so weight sizes can be compared directly.

In [6]:
for role, r in report.items():
    if 'weights' not in r: continue
    top = sorted(((k, v) for k, v in r['weights'].items() if k != 'bias'), key=lambda kv: -abs(kv[1]))[:3]
    print(f"{role:10s} strongest: " + ",  ".join(f"{k} {v:+.2f}" for k, v in top))

driver     strongest: past_response +2.21,  corridor_density +0.34,  social_degree +0.26
rider      strongest: past_response +1.39,  recent_rides +0.60,  corridor_density +0.38
connector  strongest: past_response +0.40,  has_car +0.36,  corridor_density +0.30
referrer   strongest: corridor_density +1.20,  past_response +0.65,  recent_rides +0.41


- **past_response** is the strongest signal for drivers (+2.21), riders (+1.39) and connectors: people who answered before answer again.
- For **riders**, **recent rides** (+0.60) and **corridor density** (+0.38) come next: active people with neighbours on the same route respond more, because a nudge can actually lead to a match.
- For **drivers**, corridor density (+0.34) and social ties (+0.26) follow: a busy corridor means riders are there to take the seats.
- For **referrers**, **corridor density** leads (+1.20): people on busy routes know others who commute the same way. Car owners refer slightly less (−0.40). These are the signals the intervention engine multiplies by corridor value and fatigue when it picks who to nudge:

`priority = p(role) × corridor value × (1 − fatigue)`, with a daily budget and a 2-day cooldown.

## 5. The shipped model is reproducible

Train into a **temporary** file and compare it with `models/scorer.pkl`, the file the API loads at startup. The synthetic data is seeded, so the weights should match. To actually ship a new model, run `python -m ml.train_scorer` from `backend/`.

In [7]:
tmp = Path(tempfile.mkdtemp()) / "scorer.pkl"
save(models, report, tmp)
shipped = json.loads((BACKEND / "models" / "scorer.json").read_text())
print("shipped model trained at:", shipped["trained_at"], "| data:", shipped["data"])
same = all(report[r].get("weights") == shipped["report"][r].get("weights") and report[r].get("shipped") == shipped["report"][r].get("shipped")
           for r in ROLES)
print("this run reproduces the shipped weights:", same)
print("temporary artifact:", tmp.stat().st_size, "bytes (models/scorer.pkl left unchanged)")

shipped model trained at: 2026-10-15T07:16:09.373925+00:00 | data: synthetic (labelled)
this run reproduces the shipped weights: True
temporary artifact: 2153 bytes (models/scorer.pkl left unchanged)


## 6. How the API loads the model

`scorer.load_pretrained()` reads `scorer.pkl` and turns each shipped model into one weight vector: scikit-learn keeps the intercept separately, so it's **folded into the `bias` weight** (the constant-1 feature). Roles that weren't shipped (connector) keep `DEFAULT_WEIGHTS`. Each community then gets its **own copy** of these weights (`ModelState` rows), which online learning updates.

In [8]:
from app.engine import scorer
scorer.load_pretrained.cache_clear()
w = scorer.load_pretrained(str(tmp))
for role in ROLES:
    src = "trained" if role in models else "default (not shipped)"
    print(f"{role:10s} {src:22s} " + " ".join(f"{v:+.2f}" for v in w[role]))
# a member with a car, a busy corridor, two rides last week, some ties and a good answer record
x = [1.0, 0.8, 0.5, 0.4, 0.75, 1.0]
print("\nexample member:", dict(zip(FEATURES, x)))
print("p(role):", {r: round(scorer.predict(w[r], x), 2) for r in ROLES})

driver     trained                -0.00 +0.34 +0.14 +0.26 +2.21 -2.09
rider      trained                +0.00 +0.38 +0.60 +0.22 +1.39 -1.89
connector  default (not shipped)  +0.00 +0.10 +0.10 +0.50 +0.20 -1.60
referrer   trained                -0.40 +1.20 +0.41 -0.00 +0.65 -2.56

example member: {'has_car': 1.0, 'corridor_density': 0.8, 'recent_rides': 0.5, 'social_degree': 0.4, 'past_response': 0.75, 'bias': 1.0}
p(role): {'driver': 0.5, 'rider': 0.46, 'connector': 0.25, 'referrer': 0.21}


## 7. Online learning after every outcome

After each nudge answer, ride and rating, the engine runs one logistic-regression gradient step on that community's weights:

`w ← w + lr_t × sample_weight × (y − p) × x`,  with  `lr_t = lr / (1 + updates / 1000)`

The step **shrinks as the model sees more outcomes** (`FELLARIDE_LEARNING_RATE_DECAY_UPDATES`), so a few noisy answers late on can't swing a mature model. Below, a rider model sees the same member respond 5 times, then ignore 5 times.

In [9]:
from types import SimpleNamespace
state = SimpleNamespace(weights=list(w["rider"]), updates=0)   # stands in for a ModelState row
x = [0.0, 0.7, 0.25, 0.3, 0.5, 1.0]
print(f"start         p={scorer.predict(state.weights, x):.3f}")
for i, y in enumerate([1]*5 + [0]*5, 1):
    before = scorer.learn(state, x, y, lr=0.35)
    print(f"update {i:2d} y={y}  p before={before:.3f}  after={scorer.predict(state.weights, x):.3f}")
state.updates = 5000
before = scorer.learn(state, x, 1, lr=0.35)
print(f"\nafter 5000 updates, one more 'yes' moves p only {before:.3f} -> {scorer.predict(state.weights, x):.3f} (learning rate has decayed)")

start         p=0.328
update  1 y=1  p before=0.328  after=0.432
update  2 y=1  p before=0.432  after=0.526
update  3 y=1  p before=0.526  after=0.603
update  4 y=1  p before=0.603  after=0.664
update  5 y=1  p before=0.664  after=0.711
update  6 y=0  p before=0.711  after=0.606
update  7 y=0  p before=0.606  after=0.508
update  8 y=0  p before=0.508  after=0.425
update  9 y=0  p before=0.425  after=0.359
update 10 y=0  p before=0.359  after=0.307

after 5000 updates, one more 'yes' moves p only 0.307 -> 0.323 (learning rate has decayed)


## 8. Training on real data

`python -m ml.train_scorer --source real` reads **answered nudges from real (non-synthetic) communities** in `FELLARIDE_DATABASE_URL`, pooled across communities.

- **No leakage:** the features were stored on each nudge row *when it was sent*, before the answer.
- **Time-based holdout:** rows are kept in time order and the **newest 25%** is the test set: train on the past, test on the future.
- **Labels keep coming:** unanswered nudges expire as a "no" when the cooldown ends, and 15% of each day's nudges go to random members (`explore`), so the data isn't only people the model already likes.
- The **referrer** role has no real rows yet: real members' features aren't stored at a point before they referred.

Below, the real-data loader runs against the database this notebook sees (the local development database unless `FELLARIDE_DATABASE_URL` is set). With few real answers it just reports that, as expected.

In [10]:
real = load_real_logs()
print({role: len(rows) for role, rows in real.items()}, "answered real nudges per role")
_, real_report = train(real, by_time=True)
for role, r in real_report.items():
    print(f"{role:10s}", "not enough rows yet (needs 30+, both outcomes)" if r.get("skipped") else
          f"shipped={r['shipped']} samples={r['samples']} holdout_auc={r['holdout_auc']} split={r['split']}")

{'driver': 0, 'rider': 0, 'connector': 0, 'referrer': 0} answered real nudges per role
driver     not enough rows yet (needs 30+, both outcomes)
rider      not enough rows yet (needs 30+, both outcomes)
connector  not enough rows yet (needs 30+, both outcomes)
referrer   not enough rows yet (needs 30+, both outcomes)


In [11]:
# the time-based split on the synthetic rows, to show how it differs from the random split
_, time_report = train(logs, by_time=True)
for role in ROLES:
    print(f"{role:10s} holdout AUC  random split={report[role]['holdout_auc']}   time split={time_report[role]['holdout_auc']}")

driver     holdout AUC  random split=0.776   time split=0.739
rider      holdout AUC  random split=0.712   time split=0.656
connector  holdout AUC  random split=0.6   time split=0.727
referrer   holdout AUC  random split=0.626   time split=0.659


## 9. Inference on sample members

A fresh synthetic community, 10 days through the engine, then a few members scored with the model, exactly as `GET /api/v1/tenants/{id}/members/{id}` does.

In [12]:
from sqlalchemy.orm import sessionmaker
from app.db import make_engine, init_db
from app.sim.synthetic import create_synthetic_tenant
from app.sim.behaviour import simulate
from app.engine.context import build_context

eng = make_engine("sqlite://"); init_db(eng)
db = sessionmaker(bind=eng, expire_on_commit=False)()
t = create_synthetic_tenant(db, name="inference demo", seed=999); db.commit()
simulate(db, t, 10, seed=5)
ctx = build_context(db, t)
weights = scorer.tenant_weights(db, t)
sample = [m for m in ctx.members.values() if m.registered_day is not None][:8]
print(f"{'member':14s} {'car':>3s} {'state':11s} " + " ".join(f"{r:>9s}" for r in scorer.ROLES))
for m in sample:
    s = scorer.score_member(weights, ctx.features(m))
    print(f"{m.name:14s} {'yes' if m.has_car else 'no':>3s} {m.state:11s} " + " ".join(f"{s[r]:9.2f}" for r in scorer.ROLES))

member         car state          driver     rider connector  referrer
Kabir T.        no registered       0.21      0.43      0.28      0.12
Saanvi B.       no referrer         0.16      0.44      0.30      0.10
Kavya J.       yes registered       0.09      0.28      0.18      0.09
Nikhil M.       no registered       0.22      0.43      0.28      0.14
Riya P.        yes registered       0.09      0.33      0.20      0.07
Pooja P.        no registered       0.23      0.38      0.24      0.14
Karan E.        no referrer         0.30      0.52      0.31      0.18
Harsh B.        no registered       0.10      0.36      0.27      0.09


## 10. Does it change outcomes? Smart engine vs random nudges

Same synthetic community, same seeds, 20 days, two strategies. (The README's headline numbers use the longer 40-day comparison, `POST /api/v1/demo/compare`, shown on the dashboard's Proof page.)

In [13]:
from app.engine import metrics
for strategy in ("smart", "random"):
    d = sessionmaker(bind=eng, expire_on_commit=False)()
    tt = create_synthetic_tenant(d, name=strategy, seed=42); d.commit()
    simulate(d, tt, 20, strategy=strategy, seed=3)
    k = metrics.kpis(d, tt)
    print(f"{strategy:6s} active={k['active_users']:3d} rides={k['rides_completed']:4d} match_rate={k['match_rate']:.2f} registered={k['registered']}")

smart  active= 30 rides=  60 match_rate=0.70 registered=76


random active= 13 rides=  14 match_rate=0.35 registered=56


## 11. The expansion engine's bandit

The Network Expansion Engine tries growth actions (recruit drivers, fill empty seats, invite neighbours, win back riders) and judges each one 2 days later. Per action type it keeps a **Beta(1, 1) posterior**: the success rate is `(successes + 1) / (trials + 2)`. The next action is ranked by

`opportunity × gap severity × (success rate + 0.25 × √(ln(total + 1) / (trials + 1)))`

so actions that worked are preferred, and actions tried less often get an exploration bonus (an upper-confidence-bound idea).

In [14]:
import math
def rate(s, n): return (s + 1) / (n + 2)
def bonus(n, total): return 0.25 * math.sqrt(math.log(total + 1) / (n + 1))
history = {"recruit_drivers": (3, 4), "activate_riders": (1, 5), "invite_neighbours": (0, 0)}   # (successes, trials)
total = sum(n for _, n in history.values()) + 1
for kind, (s, n) in history.items():
    print(f"{kind:16s} {s}/{n} worked  posterior mean={rate(s, n):.2f}  explore bonus={bonus(n, total):.2f}  score={rate(s, n) + bonus(n, total):.2f}")

recruit_drivers  3/4 worked  posterior mean=0.67  explore bonus=0.17  score=0.84
activate_riders  1/5 worked  posterior mean=0.29  explore bonus=0.16  score=0.44
invite_neighbours 0/0 worked  posterior mean=0.50  explore bonus=0.39  score=0.89


The untried action gets a fair chance from its prior (0.5) plus the largest bonus. The one that worked 3 of 4 times leads, and the one that failed 4 of 5 times drops behind.

## 12. Limitations

- **Synthetic labels:** the training rows and the smart-vs-random comparison come from our own behaviour model, so they show the **pipeline works**, not how real people behave. The real-data path (section 8) and the dashboard's real/synthetic split (`by_source`) are there for that next step.
- **Small real data:** until a community has a few hundred answered nudges, the shipped synthetic prior plus online learning carry the model.
- **Connector role** stays on default weights until there are enough samples.
- The shipped models' holdout AUCs (0.63 to 0.78) are moderate. That's expected with 6 coarse features, and the models are only used to **rank** whom to nudge first, not to make yes/no decisions about people.